<a href="https://colab.research.google.com/github/gokulkrishnanaofficial/Python_AI-Basic/blob/main/Game_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pycocotools #needed to get webcam input
!pip install ffmpeg

Mount Drive and define work dir

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
work_dir =  "/content/drive/My Drive/SKILLIT Courses/AI Level 2/Final Project"
os.chdir(work_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Imports

In [ ]:
from colab_utils import imshow, videoGrabber
import numpy as np
import matplotlib.pyplot as plt
from google.colab import output  # user to clear screen
%tensorflow_version 2.x
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


**Define classses**
Here we define the classes in which we are going to classify our outputs.

CLASS_NAME[0] = 'PAPER'
CLASS_NAME[1] = 'ROCK'
CLLASS_NAME[2] = 'SCISSOR'

In [ ]:
CLASS_NAME = ['Paper', 'Rock', 'Scissor']

Defin image capture function

videoGrabber()

Funtion Working


In [ ]:
def capture_images(numImages = 100, label='Null'):
 vid = videoGrabber(showVideo=True,size= (224,224))
 img=[]
 y=[]
 for x in tqdm(range(numImages)):
  new_image = np.array(vid(0))
  img.append(new_image)
  if label!= 'Null':
   y.append(label)
 img = np.array(img)
 y = np.array(y)
 return img,y

Give dataset

train_images = images/255

We conver pixels to values between 0 and 1

In [ ]:
paper_images, paper_label = capture_images(numImages=100, label=0)

  0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
rock_images, rock_label = capture_images(numImages=100, label=1)

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
scissor_images, scissor_label = capture_images(numImages=100, label=2)

In [ ]:
train_images = np.concatenate((paper_images, rock_images, scissor_images))
train_images = train_images/255
train_labels = np.concatenate((paper_label, rock_label, scissor_label))

Generating more dataset

In [ ]:
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Creates a data generator object that tranforms images
datagen = ImageDataGenerator(
rotation_range=40,
width_shift_range=0.2,
height_shift_range=0.2,
shear_range=0.2,
zoom_range=0.2,
horizontal_flip=True,
fill_mode='nearest')

idx = 0
new_train_images = []
new_train_labels = []
for image in train_images:
  img = image.reshape((1,) + image.shape)  #make the image 3D (1,60,40)
  i=0
  for batch in datagen.flow(img, save_prefix='test', save_format='jpeg'): #this loops run forever
      plt.figure()    #print (batch[0])
      plt.imshow(batch[0])
      i+=1
      new_train_images.append(batch[0])
      new_train_labels.append(train_labels[idx])
      if i > 10:
        break
  idx+=1

plt.show()

In [ ]:
import numpy as np
new_train_images = np.array(new_train_images)
new_train_labels = np.array(new_train_labels)

Using a pertrained model



picking the pre-trained model


In [ ]:
IMG_SHAPE= (224,224,3)
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')

In [ ]:
base_model.summary()

Freezing the Base

In [ ]:
base_model.trainable = False

In [ ]:
base_model.summary()

Adding the normal nerual network at the end of CNN



In [ ]:
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()

Finally, we will add the predicition layer that will be three dense neurons.

In [ ]:
prediction_layer = tf.keras.layers.Dense(3)

Noew we will combine these layers together in a model.

In [ ]:
model = tf.keras.Sequential([
  base_model,
  global_average_layer,
  prediction_layer
])

In [ ]:
model.summary()

Compliling the model


Training the model


In [ ]:
base_learning_rate = 0.0001
model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(new_train_images,new_train_labels,epochs=10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import output
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tqdm.auto import tqdm

# Imports for webcam utility using standard Colab functionality
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode
import io
from PIL import Image as PILImage

# --- Replacement for colab_utils.imshow ---
def imshow(img, title=None):
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

# --- Replacement for colab_utils.videoGrabber ---
class VideoGrabberColab:
    def __init__(self, showVideo=True, size=(224, 224)): # Changed size to match IMG_SHAPE
        # size is (width, height)
        self.width = size[0]
        self.height = size[1]

        # JavaScript code to set up the camera and capture frames
        # This code is now an immediately invoked async function that eval_js will wait for.
        js_setup_and_define = """
            (async function setupCameraAndDefineFunctions(width, height) {
                const video = document.createElement('video');
                video.style.display = 'none'; // Keep video hidden, just for capturing frames
                const stream = await navigator.mediaDevices.getUserMedia({video: {width: width, height: height}});
                document.body.appendChild(video);
                video.srcObject = stream;
                await video.play();

                const canvas = document.createElement('canvas');
                canvas.width = width;
                canvas.height = height;
                const context = canvas.getContext('2d');

                window.captureFrame = () => {
                    context.drawImage(video, 0, 0, canvas.width, canvas.height);
                    return canvas.toDataURL('image/jpeg', 0.8);
                };

                window.stopCamera = () => {
                    stream.getVideoTracks()[0].stop();
                    video.remove();
                    canvas.remove();
                };
                return true; // Signal that setup is complete
            })(%d, %d);
         """ % (self.width, self.height)

        # eval_js will block until the async JavaScript function completes
        eval_js(js_setup_and_define)
        self.started = True

    def __call__(self, timeout=0):
        if not self.started:
            raise RuntimeError("VideoGrabberColab not started. Call it with `vid = VideoGrabberColab()` first.")

        # Call the JavaScript function to capture a single frame
        data = eval_js('captureFrame()')

        # Decode the base64 image data into a numpy array
        binary_data = b64decode(data.split(',')[1])
        img_io = io.BytesIO(binary_data)
        pil_img = PILImage.open(img_io)
        # Convert to RGB if it's not (e.g., if JPEG gives grayscale sometimes)
        if pil_img.mode != 'RGB':
            pil_img = pil_img.convert('RGB')
        return np.array(pil_img) # Returns RGB image as numpy array

    def stop(self):
        # Call the JavaScript function to stop the camera and clean up
        eval_js('stopCamera()')
        self.started = False

# --- Redefine capture_images using the new VideoGrabberColab ---
def capture_images(numImages = 100, label='Null'):
    # Initialize our custom VideoGrabberColab
    vid = VideoGrabberColab(showVideo=True, size=(224, 224)) # Changed size to match IMG_SHAPE
    img = []
    y = []
    for x in tqdm(range(numImages)):
        new_image = np.array(vid(0))
        img.append(new_image)
        if label != 'Null':
            y.append(label)
    vid.stop() # Stop the webcam after capturing all images
    img = np.array(img)
    y = np.array(y)
    return img, y


Output Calculation

In [ ]:
test_image, _= capture_images(1)
test_image = test_image/255
prediction = model.predict(test_image)
plt.imshow(test_image[0])
print(CLASS_NAME[np.argmax(prediction[0])])

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32,(5,5),activation='relu',input_shape=(40,60,3)))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(64,(3,3),activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(64,(3,3),activation='relu'))

In [ ]:
model.summary()

Adding the Dense neural network at the end of CNN

In [ ]:
model.add(layers.Flatten())
model.add(layers.Dense(64,activation='relu'))
model.add(layers.Dense(3))

In [ ]:
model.summary()

Train the model

Compiling the model

Sacing the model


In [ ]:
model.save('self-made_model.h5')